# BITS PILANI WILP
## Course: Software Engineering for Machine Learning (AIMLCZG546)
### Assignment I: Requirements Engineering and System Architecture for ML

**Group Number:** 84  
**Date:** June 8, 2026

#### **Group Members & Contributions:**

| Sl. No | BITS ID | Name | Qualitative Contribution | Quantitative Contribution (%) |
|---|---|---|---|---|
| 1 | **2025AA05710** | Singh Pritesh Sanjay Poonam | Quality Attribute Testing, Performance Benchmarking, and System Integration | 25% |
| 2 | **2025AA05368** | Gangera Tushar Kantibhai Dayaben | Requirements Formulation, Problem Statement definition, and GR4ML Modeling | 25% |
| 3 | **2025AB05154** | Gangam Shuba Nandini | Feature Engineering, ML Pipeline Design, and Model Training | 25% |
| 4 | **2025AA05574** | Shaifali Garg | System Architecture Diagram, FastAPI Serving, and Event Simulation | 25% |

---


## Objective 1: Requirements Formulation

### 1. Domain & Problem Statement
* **Domain:** Consumer Credit Risk Assessment & Automated Loan Underwriting.
* **Problem Statement:** 
  Lending institutions process thousands of loan applications daily. Manual review of these applications is slow, costly, and prone to subjective biases. We design a real-time, Machine Learning-based decision support system to automate the underwriting process by predicting the **default risk** of each loan applicant. Given an applicant's demographic details, credit profile, and asset-liability levels, the system must predict if the application should be approved or denied, while assigning a structured risk tier (LOW, MEDIUM, or HIGH) and maintaining a latency of < 150ms to ensure a seamless client experience.


### 2. GR4ML Requirement Specifications & Goals

**GR4ML** (Goal-Oriented Requirements Engineering for Machine Learning) organizes our specifications into three views:

#### **A. Business View**
Aligns the high-level business goals with the ML requirements.
* **Actors:** Credit Officer (reviews flagged exceptions), Loan Applicant (submits transaction data).
* **Strategic Goals:** Minimize default losses (Risk Control), automate underwriting (Operational Efficiency).
* **Decision Goals:** Approve, Deny, or Flag application for manual audit.
* **Question Goals:** Is the credit risk acceptable? Does the client meet credit length requirements?
* **Indicators:** Default Rate < 2.5%, Auto-approval rate > 80%, response latency < 150ms.
* **Insights:** Default probability score, risk tier (LOW/MEDIUM/HIGH), computed net worth.

#### **B. Analytics Design View**
Translates business questions into algorithm selection and softgoals.
* **Analytics Goal:** Prediction (Binary classification of default risk).
* **Algorithms:** RandomForestClassifier (robust baseline) and XGBoost (advanced modeling).
* **Softgoals:** Prediction Accuracy, Inference Performance (Latency), Explainability (SHAP/Feature Importance), and Input Reliability (Type Validation).

#### **C. Data Preparation View**
Specifies the transformation workflows required to convert raw data into model features.
* **Raw Entities:** Loan Applications Table, Credit History Bureau Table, Asset Registry Table.
* **Prep Tasks:** Missing value imputation, Categorical variable encoding, Feature engineering (Net Worth, Debt-to-Income), Numerical scaling.
* **Operators:** `SimpleImputer`, `StandardScaler`, `SMOTE` (imbalance handling), and `Pydantic` validation.


### 3. GR4ML Graphical Notations

Below are the programmatically generated GR4ML modeling diagrams mapping the three system views:

#### **I. Business View Diagram**
![Business View](diagrams/gr4ml_business_view.png)

#### **II. Analytics Design View Diagram**
![Analytics Design View](diagrams/gr4ml_analytics_design_view.png)

#### **III. Data Preparation View Diagram**
![Data Prep View](diagrams/gr4ml_data_prep_view.png)


### 4. Top Three Quality Requirements

We identify the following three non-functional quality requirements as critical for the loan underwriting system:

1. **Robustness (Data Validation & Boundary Protection)**
   * **Justification:** ML models fail silently when fed out-of-distribution or corrupted data. By enforcing strict boundaries (e.g. Credit score must be 300-850, age must be >= 18) via Pydantic schemas, we protect the downstream estimator from garbage-in-garbage-out behavior.
   * **Measurable Metric:** 100% of invalid data payloads rejected with explicit validation errors before model execution.

2. **Reliability (Deterministic Tiers & Fault Tolerance)**
   * **Justification:** Loan decisions are subject to strict financial regulations. The system must output consistent classification probability bounds and structured risk categories (LOW/MEDIUM/HIGH) without crash risk under high concurrent loads.
   * **Measurable Metric:** Service availability uptime >= 99.99%, zero unhandled 500 server errors on model predictions.

3. **Performance (Low Latency Inference)**
   * **Justification:** Real-time loan decisioning is embedded directly in client-facing online application portals. High inference latencies directly lead to application drop-offs and poor user engagement.
   * **Measurable Metric:** Average model prediction pipeline response time < 150ms (SLA bound).


## Objective 2: System Architecture

### 5. System Architecture Diagram

The system architecture combines **Sculley's "Hidden Technical Debt" layout** (separating configuration, logging, serving infrastructure, and monitoring) with the **Pipe-and-Filter execution pipeline**:

![System Architecture](diagrams/system_architecture.png)

---

### 6 & 7. Selection & Implementation of Two Architectural Patterns

#### **Pattern 1: Pipe-and-Filter Pattern**
* **Application:** Implemented in `app/pipeline.py`. The prediction sequence is structured as a series of isolated filters that consume a specific input type, perform transformations, and pipe their output to the next stage.
  * **Filter 1 (validate_input):** Enforces business rules (Robustness).
  * **Filter 2 (extract_features):** Computes financial ratios and encodes features (Maintainability).
  * **Filter 3 (run_model):** Performs RandomForest prediction scoring (Performance).
  * **Filter 4 (format_response):** Maps probabilities to risk tiers (Reliability).

#### **Pattern 2: Microservices Serving / Event-Driven Logging Pattern**
* **Application:** Implemented in `app/main.py` and `app/logger.py`. The ML model is served via a FastAPI containerized microservice exposing REST APIs. The service uses asynchronous event logging with structured JSON payloads containing runtime attributes (latency, decision, features) for downstream log aggregators, and includes a `/health` endpoint to monitor model load status.


## 8. ML Pipeline Code Execution & Verification

In [1]:
# Load the generated synthetic dataset and display feature summaries
import pandas as pd
import numpy as np
import os

data_path = r"data/loan_data.csv"
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print("Dataset Shape:", df.shape)
    print("\nFirst 5 records:")
    display(df.head())
    print("\nTarget Class Distribution (loan_approved):")
    print(df['loan_approved'].value_counts(normalize=True))
else:
    print("Loan data not found. Please run the generation script first.")


Dataset Shape: (5000, 11)

First 5 records:


,age,annual_income,credit_score,loan_amount,loan_duration,savings_balance,total_assets,total_liabilities,previous_defaults,cc_utilization,loan_approved
0,56,73454.18,693,36459.00,36,32274.29,211729.10,23321.75,1,0.8725,0
1,69,171596.75,750,37153.53,24,63573.73,642292.08,148589.10,0,0.6690,1
2,46,173781.18,611,98169.08,60,136488.69,294830.31,48792.36,0,0.1211,0
3,32,117449.92,442,62684.38,60,49578.52,324972.06,77633.76,0,0.5071,0
4,60,168228.95,606,18354.22,12,110750.98,481259.34,99686.45,0,0.7425,1



Target Class Distribution (loan_approved):
loan_approved
1    0.5716
0    0.4284
Name: proportion, dtype: float64


In [2]:
# Run step 1: train baseline model and evaluate performance
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

features = [
    'age', 'annual_income', 'credit_score', 'loan_amount', 
    'loan_duration', 'savings_balance', 'total_assets', 
    'total_liabilities', 'previous_defaults', 'cc_utilization'
]
X = df[features]
y = df['loan_approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training baseline Random Forest Model...")
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print("\nAccuracy:", round(accuracy_score(y_test, preds), 4))
print("\nClassification Report:")
print(classification_report(y_test, preds))

# Save for serving
os.makedirs("app", exist_ok=True)
joblib.dump(model, "app/model.pkl")
print("Saved trained model to app/model.pkl")


Training baseline Random Forest Model...



Accuracy: 0.716

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.57      0.63       428
           1       0.72      0.82      0.77       572

    accuracy                           0.72      1000
   macro avg       0.71      0.70      0.70      1000
weighted avg       0.71      0.72      0.71      1000

Saved trained model to app/model.pkl


In [3]:
# Execute Step 2: MLflow logging simulation
# Demonstrates MLOps instrumentation for parameters, metrics, and models
from training.step2_mlflow_train import train_with_mlflow

train_with_mlflow(n_estimators=150, max_depth=6)


[WARNING] mlflow is not installed. Running in SIMULATED MLflow tracking mode.

[STEP 2] Training Model with MLflow Tracking (n_estimators=150, max_depth=6)...


--- SIMULATED MLFLOW LOGGING ---
Setting experiment: 'loan_approval_risk_experiment'
Active Run started.
Logged Parameter: n_estimators -> 150
Logged Parameter: max_depth -> 6
Logged Metric: accuracy -> 0.7210
Logged Metric: precision -> 0.7223
Logged Metric: recall -> 0.8322
Logged Metric: f1_score -> 0.7734
Logged Artifact: sklearn model -> 'loan_risk_model.pkl'
Active Run ended.
--------------------------------
Model saved successfully to c:\Users\singh\OneDrive\Documents\BITS WILP\Sem 2\SEML\SEML Assignment 1\app\model.pkl


In [4]:
# Demonstrating Pipe-and-Filter execution programmatically
from app.schemas import LoanApplicationInput
from app.pipeline import execute_pipeline
import joblib

# Load trained estimator
clf = joblib.load("app/model.pkl")

# Test cases representing low-risk and high-risk applicants
low_risk_app = LoanApplicationInput(
    age=40, annual_income=120000.0, credit_score=780, loan_amount=15000.0,
    loan_duration=24, savings_balance=50000.0, total_assets=250000.0,
    total_liabilities=10000.0, previous_defaults=0, cc_utilization=0.10
)

high_risk_app = LoanApplicationInput(
    age=22, annual_income=25000.0, credit_score=520, loan_amount=50000.0,
    loan_duration=60, savings_balance=200.0, total_assets=1000.0,
    total_liabilities=8000.0, previous_defaults=1, cc_utilization=0.95
)

print("--- EXECUTING PIPE-AND-FILTER FOR LOW-RISK APPLICANT ---")
result_low = execute_pipeline(low_risk_app, clf)
print(result_low.model_dump_json(indent=2))

print("\n--- EXECUTING PIPE-AND-FILTER FOR HIGH-RISK APPLICANT ---")
try:
    # This should fail validation because loan amount exceeds 500% of income ($50k > 5 * $25k = $125k is not exceeded, wait, $50k is 200% of $25k. Wait, let's trigger the 500% rule: loan $150k for $25k income)
    high_risk_app.loan_amount = 150000.0
    result_high = execute_pipeline(high_risk_app, clf)
    print(result_high.model_dump_json(indent=2))
except ValueError as e:
    print(f"Caught expected business rule validation exception: {str(e)}")


Configurations loaded successfully from c:\Users\singh\OneDrive\Documents\BITS WILP\Sem 2\SEML\SEML Assignment 1\configs\config.yaml
--- EXECUTING PIPE-AND-FILTER FOR LOW-RISK APPLICANT ---
{
  "is_approved": true,
  "probability": 0.9112,
  "risk_tier": "LOW",
  "net_worth": 240000.0,
  "debt_to_income": 0.0833,
  "latency_ms": 10.04
}

--- EXECUTING PIPE-AND-FILTER FOR HIGH-RISK APPLICANT ---
Caught expected business rule validation exception: Requested loan amount exceeds 500% of the applicant's annual income.


C:\Users\singh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [5]:
# Verify the FastAPI Serving infrastructure using FastAPI TestClient
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

print("--- TESTING GET /health ---")
response_health = client.get("/health")
print("Status Code:", response_health.status_code)
print("Response:", response_health.json())

print("\n--- TESTING POST /predict (Valid Payload) ---")
payload_valid = {
    "age": 35,
    "annual_income": 80000.0,
    "credit_score": 710,
    "loan_amount": 25000.0,
    "loan_duration": 36,
    "savings_balance": 12000.0,
    "total_assets": 85000.0,
    "total_liabilities": 20000.0,
    "previous_defaults": 0,
    "cc_utilization": 0.35
}
response_pred = client.post("/predict", json=payload_valid)
print("Status Code:", response_pred.status_code)
print("Response JSON:")
print(response_pred.json())

print("\n--- TESTING POST /predict (Invalid Schema Payload) ---")
# Sending an invalid credit score of 950 (must be <= 850)
payload_invalid = payload_valid.copy()
payload_invalid["credit_score"] = 950
response_invalid = client.post("/predict", json=payload_invalid)
print("Status Code (Expected 422 Unprocessable):", response_invalid.status_code)
print("Response Detail:")
print(response_invalid.json()["detail"][0]["msg"])


Model loaded successfully from c:\Users\singh\OneDrive\Documents\BITS WILP\Sem 2\SEML\SEML Assignment 1\app/model.pkl
--- TESTING GET /health ---
Status Code: 200
Response: {'status': 'healthy', 'model_status': 'loaded', 'environment': 'production', 'features_configured': 10}

--- TESTING POST /predict (Valid Payload) ---
{"timestamp": "2026-06-08T23:12:11Z", "level": "INFO", "message": "loan_evaluation_served", "logger": "loan_service", "taskName": null, "latency_ms": 3.99, "credit_score": 710, "loan_amount": 25000.0, "is_approved": true, "probability": 0.7059, "risk_tier": "LOW"}


Status Code: 200
Response JSON:
{'is_approved': True, 'probability': 0.7059, 'risk_tier': 'LOW', 'net_worth': 65000.0, 'debt_to_income': 0.25, 'latency_ms': 3.96}

--- TESTING POST /predict (Invalid Schema Payload) ---
Status Code (Expected 422 Unprocessable): 422
Response Detail:
Input should be less than or equal to 850


C:\Users\singh\AppData\Local\Programs\Python\Python313\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
C:\Users\singh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [6]:
# Run Quality Attribute Unit Tests using python's unittest runner
import unittest
from tests.test_quality_attrs import TestQualityAttributes

# Create test suite
suite = unittest.TestLoader().loadTestsFromTestCase(TestQualityAttributes)
runner = unittest.TextTestRunner(verbosity=2)
print("Running Quality Attribute Test Suite...")
runner.run(suite)


test_feature_extraction_filter_isolation (tests.test_quality_attrs.TestQualityAttributes.test_feature_extraction_filter_isolation) ... 

ok


test_inference_pipeline_latency (tests.test_quality_attrs.TestQualityAttributes.test_inference_pipeline_latency) ... 

C:\Users\singh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\singh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\singh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\singh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Running Quality Attribute Test Suite...


...

test_validation_filter_is_pure (tests.test_quality_attrs.TestQualityAttributes.test_validation_filter_is_pure) ... 

ok


----------------------------------------------------------------------
Ran 7 tests in 0.481s

OK



[PERFORMANCE] Average Pipeline Execution Latency: 5.2284 ms


<unittest.runner.TextTestResult run=7 errors=0 failures=0>

### Summary & Conclusion

Through this assignment, we successfully engineered a production-grade machine learning system:
1. **Requirements Formulation:** Developed goals and indicators using the **GR4ML** conceptual framework, mapping the Business, Analytics, and Data Preparation views.
2. **System Architecture Design:** Integrated Sculley's MLOps framework (separating data validation, model tracking, configuration, and JSON logging) with a **Pipe-and-Filter pattern** for deterministic transaction execution.
3. **Execution & Validation:** Built a FastAPI server to serve predictions and verified system stability across 4 quality attributes: **Robustness** (boundary protection), **Reliability** (consistent schemas), **Performance** (latency < 10ms), and **Maintainability** (pure isolated filters).
